# Skin Cancer Classifier — DenseNet121 on ISIC 2019

End-to-end pipeline for 8-class skin lesion classification using the ISIC 2019 dataset.

| Section | Description |
|---|---|
| 1. Preprocessing | Stratified 70 / 15 / 15 split saved to `/kaggle/working/splits/` |
| 2. Dataset | `SkinLesionDataset` with training augmentation |
| 3. Model | Pretrained DenseNet121 with custom 8-class head |
| 4. Training | Two-phase training + early stopping; best model → `/kaggle/working/best_model.pth` |
| 5. Evaluation | Per-class AUC-ROC, confusion matrix, melanoma sensitivity/specificity |
| 6. FastAPI | Inference endpoint (for local / cloud deployment after training) |

**Expected dataset layout** (`/kaggle/input/isic-2019/`):
```
isic-2019/
├── ISIC_2019_Training_Input/   ← flat folder of *.jpg images
└── ISIC_2019_Training_GroundTruth.csv
```
The CSV has columns: `image, MEL, NV, BCC, AK, BKL, DF, VASC, SCC, UNK` (one-hot).
A class-subdirectory layout (`isic-2019/{class_name}/*.jpg`) is also auto-detected.

In [ ]:
import csv
import io
import os
import random
import time
import warnings
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    auc,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.models import DenseNet121_Weights

warnings.filterwarnings("ignore")

# ── Kaggle paths ──────────────────────────────────────────────────────────────
ISIC_DIR        = Path("/kaggle/input/isic-2019")
SPLITS_DIR      = Path("/kaggle/working/splits")
WORKING_DIR     = Path("/kaggle/working")
BEST_MODEL_PATH = WORKING_DIR / "best_model.pth"

# ── Training hyperparameters ──────────────────────────────────────────────────
BATCH_SIZE   = 32
EPOCHS_1     = 10   # Phase 1: head only (frozen backbone)
EPOCHS_2     = 20   # Phase 2: full fine-tune
LR_1         = 1e-3
LR_2         = 1e-4
PATIENCE     = 5
NUM_WORKERS  = 2
SEED         = 42

# ── Class definitions ─────────────────────────────────────────────────────────
CLASSES     = ["melanoma", "nevus", "bcc", "ak", "bkl", "df", "vasc", "scc"]
LABEL2IDX   = {cls: i for i, cls in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

# Map ISIC 2019 one-hot column names → our canonical class names
ISIC_COL_MAP = {
    "MEL": "melanoma", "NV": "nevus",  "BCC": "bcc",
    "AK":  "ak",       "BKL": "bkl",  "DF":  "df",
    "VASC": "vasc",    "SCC": "scc",
}

# ── Device ────────────────────────────────────────────────────────────────────
def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = get_device()
print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")

SPLITS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Preprocessing — Stratified Splits (70 / 15 / 15)

Reads images and labels from `/kaggle/input/isic-2019/`, then writes
`train.csv`, `val.csv`, and `test.csv` to `/kaggle/working/splits/`.

Each CSV row: `image_path, label_idx, label_name`.

In [ ]:
# ── Sample collection ────────────────────────────────────────────────────────

def _collect_from_gt_csv(isic_dir: Path) -> list[tuple[str, int]]:
    """ISIC 2019 layout: flat image folder + GroundTruth CSV."""
    gt_csv  = isic_dir / "ISIC_2019_Training_GroundTruth.csv"
    img_dir = isic_dir / "ISIC_2019_Training_Input"
    samples = []
    with open(gt_csv, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            image_id = row["image"].strip()
            label_name = None
            for col, cls in ISIC_COL_MAP.items():
                val = row.get(col, "0").strip()
                if val in ("1", "1.0"):
                    label_name = cls
                    break
            if label_name is None:
                continue
            for ext in (".jpg", ".JPG", ".jpeg", ".png"):
                img_path = img_dir / f"{image_id}{ext}"
                if img_path.exists():
                    samples.append((str(img_path), LABEL2IDX[label_name]))
                    break
    return samples


def _collect_from_subdirs(isic_dir: Path) -> list[tuple[str, int]]:
    """Class-subdirectory layout: isic_dir/{class_name}/*.jpg"""
    samples = []
    for cls in CLASSES:
        cls_dir = isic_dir / cls
        if not cls_dir.is_dir():
            print(f"  Warning: directory not found: {cls_dir}")
            continue
        for img_path in sorted(cls_dir.iterdir()):
            if img_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                samples.append((str(img_path), LABEL2IDX[cls]))
    return samples


def collect_samples(isic_dir: Path) -> list[tuple[str, int]]:
    gt_csv = isic_dir / "ISIC_2019_Training_GroundTruth.csv"
    if gt_csv.exists():
        print("Layout: ISIC 2019 flat (CSV + image folder)")
        return _collect_from_gt_csv(isic_dir)
    print("Layout: class subdirectories")
    return _collect_from_subdirs(isic_dir)


# ── Stratified split ──────────────────────────────────────────────────────────

def stratified_split(
    samples: list[tuple[str, int]],
    train_frac: float = 0.70,
    val_frac: float   = 0.15,
    seed: int = SEED,
) -> tuple[list, list, list]:
    rng = random.Random(seed)
    by_class: dict[int, list] = defaultdict(list)
    for s in samples:
        by_class[s[1]].append(s)

    train, val, test = [], [], []
    for label_idx, class_samples in sorted(by_class.items()):
        rng.shuffle(class_samples)
        n       = len(class_samples)
        n_train = max(1, round(n * train_frac))
        n_val   = max(1, round(n * val_frac))
        n_train = min(n_train, n - 2) if n >= 3 else n_train
        train.extend(class_samples[:n_train])
        val.extend(class_samples[n_train : n_train + n_val])
        test.extend(class_samples[n_train + n_val :])
    return train, val, test


def write_split(split: list[tuple[str, int]], path: Path) -> None:
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["image_path", "label_idx", "label_name"])
        for img_path, label_idx in split:
            writer.writerow([img_path, label_idx, CLASSES[label_idx]])


def print_split_stats(name: str, split: list[tuple[str, int]]) -> None:
    counts: dict[int, int] = defaultdict(int)
    for _, idx in split:
        counts[idx] += 1
    print(f"\n  {name} ({len(split)} images)")
    for cls in CLASSES:
        print(f"    {cls:<12} {counts[LABEL2IDX[cls]]:>5}")


# ── Run ───────────────────────────────────────────────────────────────────────

print(f"Scanning {ISIC_DIR} …")
_samples = collect_samples(ISIC_DIR)
if not _samples:
    raise RuntimeError(f"No images found in {ISIC_DIR}. Check your dataset path.")
print(f"Total images: {len(_samples)}")

_train, _val, _test = stratified_split(_samples)
for _name, _split in [("train", _train), ("val", _val), ("test", _test)]:
    print_split_stats(_name, _split)

write_split(_train, SPLITS_DIR / "train.csv")
write_split(_val,   SPLITS_DIR / "val.csv")
write_split(_test,  SPLITS_DIR / "test.csv")
print(f"\nSplits saved to {SPLITS_DIR}")

## 2. Dataset & Transforms

`SkinLesionDataset` reads the split CSVs produced above.

- **Training**: random crop, horizontal/vertical flip, rotation ±20°, colour jitter, ImageNet normalisation
- **Val / Test**: deterministic resize → centre crop → ImageNet normalisation

In [ ]:
# ImageNet statistics (DenseNet121 pretrained on ImageNet)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMAGE_SIZE    = 224


def get_transforms(split: str) -> transforms.Compose:
    if split == "train":
        return transforms.Compose([
            transforms.Resize(256),
            transforms.RandomCrop(IMAGE_SIZE),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(20),
            transforms.ColorJitter(
                brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05
            ),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


class SkinLesionDataset(Dataset):
    """
    Args:
        csv_path : Path to train.csv / val.csv / test.csv.
        split    : "train" | "val" | "test"  — controls augmentation.
    """

    def __init__(self, csv_path: Path, split: str = "train") -> None:
        self.split     = split
        self.transform = get_transforms(split)
        self.samples: list[tuple[str, int]] = []
        with open(csv_path, newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                self.samples.append((row["image_path"], int(row["label_idx"])))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int]:
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        return self.transform(image), label

    @property
    def class_counts(self) -> list[int]:
        counts = [0] * NUM_CLASSES
        for _, label in self.samples:
            counts[label] += 1
        return counts

    @property
    def class_weights(self) -> torch.Tensor:
        counts = self.class_counts
        total  = sum(counts)
        return torch.tensor(
            [total / (NUM_CLASSES * max(c, 1)) for c in counts],
            dtype=torch.float,
        )


print("SkinLesionDataset defined.")

## 3. Model — DenseNet121

Pretrained ImageNet backbone with a custom head:

```
DenseNet121 features  →  Linear(1024, 256)  →  ReLU  →  Dropout(0.4)  →  Linear(256, 8)
```

`freeze_backbone()` makes only the head trainable (Phase 1).
`unfreeze_all()` opens the full network for fine-tuning (Phase 2).

In [ ]:
def build_model(num_classes: int = NUM_CLASSES, pretrained: bool = True) -> nn.Module:
    weights = DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
    model   = models.densenet121(weights=weights)
    in_feat = model.classifier.in_features          # 1024
    model.classifier = nn.Sequential(
        nn.Linear(in_feat, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.4),
        nn.Linear(256, num_classes),
    )
    return model


def freeze_backbone(model: nn.Module) -> None:
    """Freeze everything except the classifier head (Phase 1)."""
    for name, param in model.named_parameters():
        param.requires_grad = name.startswith("classifier")


def unfreeze_all(model: nn.Module) -> None:
    """Unfreeze every parameter (Phase 2 fine-tuning)."""
    for param in model.parameters():
        param.requires_grad = True


def count_trainable(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Sanity-check parameter counts
_m = build_model(pretrained=False)
freeze_backbone(_m)
print(f"Trainable params (frozen backbone): {count_trainable(_m):>10,}")
unfreeze_all(_m)
print(f"Trainable params (all unfrozen):    {count_trainable(_m):>10,}")
del _m

## 4. Training

**Phase 1** — backbone frozen, head trained with Adam (LR 1e-3), ReduceLROnPlateau scheduler.
**Phase 2** — full network fine-tuned with Adam (LR 1e-4, weight_decay 1e-4), cosine annealing.

Early stopping monitors val macro-AUC (patience = 5).
Best model (across both phases) is saved to `/kaggle/working/best_model.pth`.

In [ ]:
def make_loader(csv_path: Path, split: str) -> DataLoader:
    ds      = SkinLesionDataset(csv_path, split=split)
    shuffle = split == "train"
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS > 0,
    )


def compute_macro_auc(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            probs  = torch.softmax(model(images), dim=1).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.numpy())
    all_probs  = np.concatenate(all_probs,  axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    try:
        return float(roc_auc_score(
            all_labels, all_probs,
            multi_class="ovr", average="macro",
            labels=list(range(NUM_CLASSES)),
        ))
    except ValueError:
        return 0.0


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
) -> float:
    model.train()
    running_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)


def save_checkpoint(model: nn.Module, epoch: int, val_auc: float, path: Path) -> None:
    torch.save({
        "epoch": epoch,
        "val_auc": val_auc,
        "model_state_dict": model.state_dict(),
        "classes": CLASSES,
    }, path)
    print(f"  Checkpoint saved → {path}  (val_auc={val_auc:.4f})")


def run_phase(
    phase: int,
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scheduler,
    criterion: nn.Module,
    num_epochs: int,
    log_writer: csv.writer,
) -> float:
    best_auc, no_improve = 0.0, 0
    phase_ckpt = WORKING_DIR / f"best_phase{phase}.pth"

    for epoch in range(1, num_epochs + 1):
        t0         = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        val_auc    = compute_macro_auc(model, val_loader)
        elapsed    = time.time() - t0

        print(
            f"  Phase {phase} | Epoch {epoch:>3}/{num_epochs} "
            f"| loss {train_loss:.4f} | val_auc {val_auc:.4f} | {elapsed:.1f}s"
        )
        log_writer.writerow([phase, epoch, f"{train_loss:.6f}", f"{val_auc:.6f}"])

        if scheduler is not None:
            scheduler.step(val_auc if isinstance(scheduler,
                torch.optim.lr_scheduler.ReduceLROnPlateau) else None)

        if val_auc > best_auc:
            best_auc, no_improve = val_auc, 0
            save_checkpoint(model, epoch, val_auc, phase_ckpt)
            save_checkpoint(model, epoch, val_auc, BEST_MODEL_PATH)
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f"  Early stopping after {PATIENCE} epochs with no improvement.")
                break

    return best_auc


# ── Build loaders & model ────────────────────────────────────────────────────
train_loader = make_loader(SPLITS_DIR / "train.csv", "train")
val_loader   = make_loader(SPLITS_DIR / "val.csv",   "val")

train_ds: SkinLesionDataset = train_loader.dataset  # type: ignore[assignment]
class_weights = train_ds.class_weights.to(DEVICE)
criterion     = nn.CrossEntropyLoss(weight=class_weights)
print(f"Class weights: {[round(w, 3) for w in class_weights.cpu().tolist()]}")

model = build_model(num_classes=NUM_CLASSES, pretrained=True).to(DEVICE)

# ── Log file ──────────────────────────────────────────────────────────────────
_log_path = WORKING_DIR / "training_log.csv"
_log_file = open(_log_path, "w", newline="")
_log_writer = csv.writer(_log_file)
_log_writer.writerow(["phase", "epoch", "train_loss", "val_auc"])

# ── Phase 1: head only ────────────────────────────────────────────────────────
print("\n=== Phase 1: Training head (backbone frozen) ===")
freeze_backbone(model)
optimizer1 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR_1
)
scheduler1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer1, mode="max", factor=0.5, patience=2
)
run_phase(1, model, train_loader, val_loader, optimizer1, scheduler1,
          criterion, EPOCHS_1, _log_writer)

# ── Phase 2: full fine-tune ───────────────────────────────────────────────────
print("\n=== Phase 2: Fine-tuning full network ===")
unfreeze_all(model)
optimizer2 = torch.optim.Adam(model.parameters(), lr=LR_2, weight_decay=1e-4)
scheduler2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer2, T_max=EPOCHS_2, eta_min=1e-6
)
run_phase(2, model, train_loader, val_loader, optimizer2, scheduler2,
          criterion, EPOCHS_2, _log_writer)

_log_file.close()
print(f"\nTraining complete. Log → {_log_path}")

## 5. Evaluation

Loads the best checkpoint and evaluates on the held-out test set:

- Per-class one-vs-rest AUC-ROC
- Macro and weighted average AUC
- Confusion matrix → `/kaggle/working/confusion_matrix.png`
- Melanoma sensitivity (recall) & specificity
- Melanoma ROC curve → `/kaggle/working/roc_melanoma.png`

In [ ]:
def load_model_from_ckpt(ckpt_path: Path) -> nn.Module:
    m = build_model(num_classes=NUM_CLASSES, pretrained=False)
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
    m.load_state_dict(ckpt["model_state_dict"])
    m.to(DEVICE).eval()
    print(f"Loaded checkpoint  epoch={ckpt.get('epoch','?')}  val_auc={ckpt.get('val_auc',0):.4f}")
    return m


@torch.no_grad()
def run_inference(
    m: nn.Module, loader: DataLoader
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    all_probs, all_preds, all_labels = [], [], []
    for images, labels in loader:
        images  = images.to(DEVICE, non_blocking=True)
        logits  = m(images)
        probs   = torch.softmax(logits, dim=1).cpu().numpy()
        preds   = logits.argmax(dim=1).cpu().numpy()
        all_probs.append(probs)
        all_preds.append(preds)
        all_labels.append(labels.numpy())
    return (
        np.concatenate(all_probs,  axis=0),
        np.concatenate(all_preds,  axis=0),
        np.concatenate(all_labels, axis=0),
    )


def report_auc(all_probs: np.ndarray, all_labels: np.ndarray) -> None:
    print("\n=== Per-class AUC-ROC (one-vs-rest) ===")
    for i, cls in enumerate(CLASSES):
        bin_labels = (all_labels == i).astype(int)
        if bin_labels.sum() == 0:
            print(f"  {cls:<12}  N/A")
            continue
        print(f"  {cls:<12}  {roc_auc_score(bin_labels, all_probs[:, i]):.4f}")
    macro    = roc_auc_score(all_labels, all_probs, multi_class="ovr",
                             average="macro",    labels=list(range(NUM_CLASSES)))
    weighted = roc_auc_score(all_labels, all_probs, multi_class="ovr",
                             average="weighted", labels=list(range(NUM_CLASSES)))
    print(f"\n  Macro AUC:    {macro:.4f}")
    print(f"  Weighted AUC: {weighted:.4f}")


def plot_confusion_matrix(all_preds: np.ndarray, all_labels: np.ndarray) -> None:
    cm   = confusion_matrix(all_labels, all_preds, labels=list(range(NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(
        ax=ax, colorbar=True, xticks_rotation=45
    )
    ax.set_title("Confusion Matrix — Test Set")
    fig.tight_layout()
    out = WORKING_DIR / "confusion_matrix.png"
    fig.savefig(out, dpi=150)
    plt.show()
    print(f"Saved → {out}")


def plot_roc_melanoma(all_probs: np.ndarray, all_labels: np.ndarray) -> None:
    mel_idx  = CLASSES.index("melanoma")
    bin_lbl  = (all_labels == mel_idx).astype(int)
    fpr, tpr, _ = roc_curve(bin_lbl, all_probs[:, mel_idx])
    roc_auc_val = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc_val:.4f}")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set(xlabel="False Positive Rate", ylabel="True Positive Rate",
           title="ROC Curve — Melanoma")
    ax.legend(loc="lower right")
    fig.tight_layout()
    out = WORKING_DIR / "roc_melanoma.png"
    fig.savefig(out, dpi=150)
    plt.show()
    print(f"Saved → {out}")


def melanoma_sens_spec(
    all_preds: np.ndarray, all_labels: np.ndarray
) -> tuple[float, float]:
    mel_idx   = CLASSES.index("melanoma")
    bin_true  = (all_labels == mel_idx).astype(int)
    bin_pred  = (all_preds  == mel_idx).astype(int)
    TP = int(((bin_true == 1) & (bin_pred == 1)).sum())
    TN = int(((bin_true == 0) & (bin_pred == 0)).sum())
    FP = int(((bin_true == 0) & (bin_pred == 1)).sum())
    FN = int(((bin_true == 1) & (bin_pred == 0)).sum())
    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0
    return sensitivity, specificity


# ── Run evaluation ────────────────────────────────────────────────────────────
test_ds     = SkinLesionDataset(SPLITS_DIR / "test.csv", split="test")
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print(f"Test set: {len(test_ds)} images")

eval_model = load_model_from_ckpt(BEST_MODEL_PATH)
all_probs, all_preds, all_labels = run_inference(eval_model, test_loader)

report_auc(all_probs, all_labels)
plot_confusion_matrix(all_preds, all_labels)
plot_roc_melanoma(all_probs, all_labels)

sensitivity, specificity = melanoma_sens_spec(all_preds, all_labels)
print(f"\n=== Melanoma ===")
print(f"  Sensitivity (recall): {sensitivity:.4f}")
print(f"  Specificity:          {specificity:.4f}")

## 6. FastAPI Inference Endpoint

The cell below defines a self-contained FastAPI application for serving the trained model.
It is **not executed during Kaggle training** — copy it into a standalone `inference_api.py`
and run with:

```bash
CHECKPOINT_PATH=/kaggle/working/best_model.pth \
    uvicorn inference_api:app --host 0.0.0.0 --port 8000
```

```bash
# Test endpoint
curl -X POST "http://localhost:8000/predict" -F "file=@lesion.jpg"
```

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FastAPI inference endpoint — deploy after training             ║
# ║  pip install fastapi uvicorn python-multipart                   ║
# ╚══════════════════════════════════════════════════════════════════╝

try:
    from fastapi import FastAPI, File, HTTPException, UploadFile
    from pydantic import BaseModel as _BaseModel
    _FASTAPI_AVAILABLE = True
except ImportError:
    _FASTAPI_AVAILABLE = False
    print("fastapi not installed — skipping app definition.")

if _FASTAPI_AVAILABLE:
    _CHECKPOINT_PATH = Path(
        os.getenv("CHECKPOINT_PATH", str(BEST_MODEL_PATH))
    )
    _preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

    app = FastAPI(
        title="Skin Lesion Classifier",
        description="DenseNet121-based classifier for 8 ISIC skin lesion categories.",
        version="1.0.0",
    )

    _api_device = get_device()
    _api_model  = None

    @app.on_event("startup")
    def _startup() -> None:
        global _api_model
        if not _CHECKPOINT_PATH.exists():
            print(f"WARNING: checkpoint not found at {_CHECKPOINT_PATH}.")
            return
        _api_model = load_model_from_ckpt(_CHECKPOINT_PATH)
        print("Model loaded for inference.")

    class PredictionResponse(_BaseModel):
        predicted_class: str
        predicted_index: int
        probabilities: dict[str, float]

    class HealthResponse(_BaseModel):
        status: str
        model_loaded: bool
        device: str

    @app.get("/health", response_model=HealthResponse)
    def health() -> HealthResponse:
        return HealthResponse(
            status="ok",
            model_loaded=_api_model is not None,
            device=str(_api_device),
        )

    @app.post("/predict", response_model=PredictionResponse)
    async def predict(file: UploadFile = File(...)) -> PredictionResponse:
        if _api_model is None:
            raise HTTPException(503, f"Model not loaded ({_CHECKPOINT_PATH}).")
        if file.content_type not in ("image/jpeg", "image/png", "image/jpg"):
            raise HTTPException(415, f"Unsupported type: {file.content_type}.")
        raw = await file.read()
        if not raw:
            raise HTTPException(400, "Empty file.")
        try:
            img = Image.open(io.BytesIO(raw)).convert("RGB")
        except Exception as exc:
            raise HTTPException(400, f"Cannot decode image: {exc}") from exc
        tensor = _preprocess(img).unsqueeze(0).to(_api_device)
        with torch.no_grad():
            probs = torch.softmax(_api_model(tensor), dim=1).squeeze(0).cpu().tolist()
        pred_idx = int(np.argmax(probs))
        return PredictionResponse(
            predicted_class=CLASSES[pred_idx],
            predicted_index=pred_idx,
            probabilities={c: round(p, 6) for c, p in zip(CLASSES, probs)},
        )

    print("FastAPI app defined. Run with: uvicorn <this_module>:app --port 8000")